# Cross-Domain Insights

This notebook analyzes patterns and insights across all four domains.

## Setup

In [ ]:
import sys
sys.path.append('..')

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.cross_domain_analysis import CrossDomainAnalyzer
from src.analysis.visualization import Visualizer
from src.analysis.report_generator import ReportGenerator

## Load Results from All Domains

In [ ]:
analyzer = CrossDomainAnalyzer(results_dir="../results")
analyzer.load_results()

print("Loaded results for domains:")
for domain in analyzer.results:
    n_approaches = len([r for r in analyzer.results[domain] if r.get('success', False)])
    print(f"  - {analyzer.domain_names[domain]}: {n_approaches} approaches")

## Compare Philosophies Across Domains

In [ ]:
philosophy_df = analyzer.compare_philosophies()
philosophy_df.head(20)

## Trade-off Analysis

In [ ]:
trade_offs = analyzer.analyze_trade_offs()

# Accuracy vs Speed
acc_speed_df = pd.DataFrame(trade_offs['accuracy_vs_speed'])
acc_speed_df.head(20)

## Visualize Trade-offs

In [ ]:
plt.figure(figsize=(12, 8))

colors = {'domain_a': 'blue', 'domain_b': 'red', 
          'domain_c': 'green', 'domain_d': 'orange'}

for _, row in acc_speed_df.iterrows():
    plt.scatter(
        row['training_time'], 
        row['accuracy'],
        c=colors.get(row['domain'], 'gray'),
        s=100,
        alpha=0.7
    )
    plt.annotate(
        row['approach'][:10], 
        (row['training_time'], row['accuracy']),
        fontsize=8
    )

plt.xlabel('Training Time (s)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Training Time Across All Domains')
plt.xscale('log')

# Legend
for domain, color in colors.items():
    plt.scatter([], [], c=color, label=analyzer.domain_names.get(domain, domain))
plt.legend()
plt.show()

## Key Insights

In [ ]:
insights = analyzer.generate_insights()
print('\n'.join(insights))

## Generate Full Report

In [ ]:
report = analyzer.create_summary_report()
print(report[:3000])  # First 3000 characters

## Approach Category Analysis

In [ ]:
categories = analyzer.get_approach_categories()

print("Approach Categories:")
for category, approaches in categories.items():
    print(f"\n{category.upper()}:")
    for approach in approaches:
        print(f"  - {approach}")

## Summary Statistics

In [ ]:
summary_stats = []

for domain, results in analyzer.results.items():
    successful = [r for r in results if r.get('success', False)]
    
    if not successful:
        continue
    
    training_times = [r.get('training_time', 0) for r in successful]
    
    summary_stats.append({
        'Domain': analyzer.domain_names[domain],
        'Approaches': len(successful),
        'Avg Training Time': np.mean(training_times),
        'Min Training Time': np.min(training_times),
        'Max Training Time': np.max(training_times)
    })

pd.DataFrame(summary_stats)

## Save Report

In [ ]:
generator = ReportGenerator(results_dir="../results")
generator.save_report("../results/REPORT.md")
print("Report saved to ../results/REPORT.md")